In [1]:
import torch
import numpy as np
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cuda


- **state_dict**: The DNA of your model (weights).
- **Checkpointing**: Saving the "Universe" (Model + Optimizer + Epoch) so you can resume training after a crash.
- **Hooks**: Injecting code into the forward/backward pass without modifying the class itself (e.g., to extract feature maps from a pre-trained ResNet).

In [ ]:
# Save only the weights (Recommended)
torch.save(model.state_dict(), 'weights.pth')

# Load the weights
# 1. You must instantiate the EXACT same architecture first
model = MyModel() 
# 2. Load the dictionary
state_dict = torch.load('weights.pth')
# 3. Map dictionary to model
model.load_state_dict(state_dict)

**Why not save the whole model object?**

If you use `torch.save(model, 'model.pth')`, it uses Python "Pickle". If you change your class definition (e.g., rename a variable in `__init__`) and try to load the old file, it breaks. Always save the `state_dict`.

## 2. Checkpointing (The Time Machine)

Research experiments run for days. If your server crashes on Epoch 99 of 100, and you only saved the model weights, you have lost the Optimizer State (momentum buffers, learning rate schedules).

**To resume perfectly, you must save a custom dictionary:**

In [ ]:
# --- SAVING ---
checkpoint = {
    'epoch': 50,
    'model_state': model.state_dict(),
    'optimizer_state': optimizer.state_dict(), # Saves momentum/buffers
    'loss': 0.024
}
torch.save(checkpoint, 'checkpoint.pth')

# --- LOADING (Resuming) ---
loaded_checkpoint = torch.load('checkpoint.pth')

model.load_state_dict(loaded_checkpoint['model_state'])
optimizer.load_state_dict(loaded_checkpoint['optimizer_state'])
start_epoch = loaded_checkpoint['epoch']

print(f"Resuming training from epoch {start_epoch}")

## 3. Hooks (The X-Ray)
This is a superpower. Hooks allow you to inspect or modify data passing through the network without editing the model class code.

Use Case: You downloaded a ResNet. You want to see the output of the middle layer (feature extraction) to build a style-transfer system. You don't need to copy-paste the ResNet code and hack the forward method. You just attach a hook.

In [ ]:
# Define a dictionary to store the output
features = {}

def get_activation(name):
    # This function runs automatically during the forward pass
    def hook(model, input, output):
        features[name] = output.detach()
    return hook

# Attach the hook to a specific layer
# model.layer1 is a specific part of the network
model.layer1.register_forward_hook(get_activation('layer1'))

# Run a forward pass
output = model(input_image)

# Now 'features' contains the intermediate data!
print(features['layer1'].shape)

# Task

**Phase 1:**

1. Create a simple Model and Optimizer (SGD with momentum=0.9).
2. Train for 5 epochs.
3. Print the loss at Epoch 5.
4. Save a Checkpoint (Epoch, Model State, Optimizer State).

**Phase 2:**

1. "Restart" the script (Re-instantiate model and optimizer fresh).
2. Load the Checkpoint.
3. Train for 5 more epochs (Epochs 6-10).
4. Verify the optimizer didn't reset (e.g., if you print optimizer state, it should show history).

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import os

# --- Setup: Define a simple problem ---
# We want to learn y = 2x
X = torch.ones(10, 1)
Y = torch.ones(10, 1) * 2

class TinyModel(nn.Module):
    def __init__(self):
        super(TinyModel, self).__init__()
        self.fc = nn.Linear(1, 1, bias=False)

    def forward(self, x):
        return self.fc(x)

# File path for our checkpoint
CHECKPOINT_PATH = "research_experiment_ckpt.pth"

# ==========================================
# PHASE 1: The Initial Run (Epochs 0-4)
# ==========================================
print("--- PHASE 1: Initial Training ---")

# 1. Initialize
model_1 = TinyModel()
# Use Momentum! This has internal state that needs saving.
optimizer_1 = optim.SGD(model_1.parameters(), lr=0.1, momentum=0.9) 

# 2. Train for 5 epochs
for epoch in range(5):
    optimizer_1.zero_grad()
    output = model_1(X)
    loss = (output - Y).pow(2).mean()
    loss.backward()
    optimizer_1.step()
    
    print(f"Epoch {epoch} | Loss: {loss.item():.6f} | Weight: {model_1.fc.weight.item():.4f}")

# 3. SAVE THE CHECKPOINT
print("\n>> Simulating Crash... Saving Checkpoint...")
checkpoint = {
    'epoch': 5,                         # Save where we stopped (next epoch index)
    'model_state': model_1.state_dict(),
    'optimizer_state': optimizer_1.state_dict(),
    'loss': loss.item()
}
torch.save(checkpoint, CHECKPOINT_PATH)
print(">> Checkpoint saved successfully.\n")


# ==========================================
# PHASE 2: The Resume (Epochs 5-9)
# ==========================================
print("--- PHASE 2: Resuming from Crash ---")

# 1. Re-initialize EVERYTHING (Simulating a fresh script run)
model_2 = TinyModel()
optimizer_2 = optim.SGD(model_2.parameters(), lr=0.1, momentum=0.9)

# PROOF: Before loading, the weight is random
print(f"Checking fresh model weight (Random): {model_2.fc.weight.item():.4f}")

# 2. LOAD THE CHECKPOINT
print(">> Loading Checkpoint...")
loaded_ckpt = torch.load(CHECKPOINT_PATH)

# Restore Model Weights
model_2.load_state_dict(loaded_ckpt['model_state'])
# Restore Optimizer Momentum/Buffers
optimizer_2.load_state_dict(loaded_ckpt['optimizer_state'])
# Restore Epoch Counter
start_epoch = loaded_ckpt['epoch']

print(f"Restored model weight: {model_2.fc.weight.item():.4f}")
print(f"Resuming at Epoch: {start_epoch}\n")

# 3. Continue Training
# We continue from start_epoch (5) to 10
for epoch in range(start_epoch, 10):
    optimizer_2.zero_grad()
    output = model_2(X)
    loss = (output - Y).pow(2).mean()
    loss.backward()
    optimizer_2.step()
    
    print(f"Epoch {epoch} | Loss: {loss.item():.6f} | Weight: {model_2.fc.weight.item():.4f}")

print("\nTraining Complete.")

# Cleanup
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print(">> Cleanup: Checkpoint file removed.")

--- PHASE 1: Initial Training ---
Epoch 0 | Loss: 1.485309 | Weight: 1.0250
Epoch 1 | Loss: 0.950598 | Weight: 1.4394
Epoch 2 | Loss: 0.314292 | Weight: 1.9244
Epoch 3 | Loss: 0.005710 | Weight: 2.3761
Epoch 4 | Loss: 0.141452 | Weight: 2.7074

>> Simulating Crash... Saving Checkpoint...
>> Checkpoint saved successfully.

--- PHASE 2: Resuming from Crash ---
Checking fresh model weight (Random): 0.9241
>> Loading Checkpoint...
Restored model weight: 2.7074
Resuming at Epoch: 5

Epoch 5 | Loss: 0.500382 | Weight: 2.8640
Epoch 6 | Loss: 0.746582 | Weight: 2.8322
Epoch 7 | Loss: 0.692633 | Weight: 2.6372
Epoch 8 | Loss: 0.405989 | Weight: 2.3342
Epoch 9 | Loss: 0.111671 | Weight: 1.9946

Training Complete.
>> Cleanup: Checkpoint file removed.
Epoch 0 | Loss: 1.485309 | Weight: 1.0250
Epoch 1 | Loss: 0.950598 | Weight: 1.4394
Epoch 2 | Loss: 0.314292 | Weight: 1.9244
Epoch 3 | Loss: 0.005710 | Weight: 2.3761
Epoch 4 | Loss: 0.141452 | Weight: 2.7074

>> Simulating Crash... Saving Checkpoin